# Echo of Recitation (صدى التلاوة): Tajweed Rule Classification from Audio

Classify each Quran recitation clip into one of four rules for *noon sakinah / tanween*:
**Izhar**, **Idgham**, **Iqlab**, **Ikhfa**. Metric: macro F1.

**The trap:** test clips come from reciters who never appear in training, so a model that learns *voices* will not generalize.

**Approach:** transcribe the audio to vowelled Arabic text, then read the rule from the letter that follows the *noon sakinah* or *tanween*:
- throat letters (ء هـ ع ح غ خ) → Izhar
- (ي ر م ل و ن) → Idgham
- (ب) → Iqlab
- the remaining 15 letters → Ikhfa

Text does not depend on the reciter's voice, so the method is robust to unseen reciters.

> Run on Kaggle with **GPU T4** and **Internet on** (pretrained ASR models are downloaded from Hugging Face).

## 1. Load the data

In [ ]:
import os, glob, gc, unicodedata
import numpy as np, pandas as pd
import librosa

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
DATA = os.path.dirname(hits[0]) if hits else '.'
train  = pd.read_csv(f'{DATA}/train.csv')
test   = pd.read_csv(f'{DATA}/test.csv')
sample = pd.read_csv(f'{DATA}/sample_submission.csv')
SR = 16000
def path(split, clip_id): return f'{DATA}/{split}/{clip_id}.wav'

print(train.shape, test.shape)
print(train.label.value_counts())   # Idgham 292, Ikhfa 249, Izhar 193, Iqlab 120

## 2. Sanity check: does clip length leak the label?
Mean duration differs a little by class in training, and test durations have extreme outliers (up to ~230 s), so duration is not used as a feature.

In [ ]:
train['dur'] = [librosa.get_duration(path=path('train', i)) for i in train.id]
test['dur']  = [librosa.get_duration(path=path('test', i)) for i in test.id]
print(train.groupby('label').dur.describe().round(2))
print('test:', test.dur.describe().round(2).to_dict())

## 3. Speech-to-text with a Quran-tuned Whisper model
`tarteel-ai/whisper-base-ar-quran` outputs vowelled Quranic text (including sukun and tanween marks), which is exactly what the rule logic needs.

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
asr = pipeline('automatic-speech-recognition', model='tarteel-ai/whisper-base-ar-quran', device=device,
               torch_dtype=torch.float16 if device == 0 else torch.float32)

def transcribe(split, ids, bs=16):
    out, ids = [], list(ids)
    for s in range(0, len(ids), bs):
        batch = [{'raw': librosa.load(path(split, i), sr=SR)[0], 'sampling_rate': SR} for i in ids[s:s+bs]]
        out += [r['text'] for r in asr(batch, batch_size=bs, chunk_length_s=30)]
    return out

train['text'] = transcribe('train', train.id)
test['text']  = transcribe('test', test.id)
for lab, g in train.groupby('label'):
    print(lab, g.text.head(3).tolist())

## 4. Tajweed rule engine
Tokenize the text into letters with their diacritics, find every *noon sakinah* / *tanween*, look at the next letter and count evidence for each rule.

Details handled:
- tanween followed by a silent alif/ya (e.g. ـًا) skips to the next real letter
- a small meem mark on noon is direct evidence of Iqlab
- meem sakinah followed by ب at a word start is counted as Iqlab evidence
- an alif at the start of a word is treated as hamza (Izhar)

Each kind of evidence gets a weight; the rule with the highest score wins. If nothing is found, the clip is left for the model in step 5.

In [ ]:
DIAC = set(chr(c) for c in list(range(0x0610,0x061B)) + list(range(0x064B,0x0660)) + [0x0670]
           + list(range(0x06D6,0x06EE)) + list(range(0x08D3,0x0900))) | {'\u0640'}
TANWEEN    = {'\u064B','\u064C','\u064D','\u08F0','\u08F1','\u08F2'}
VOWELS     = {'\u064E','\u064F','\u0650','\u0651'}
SUKUN      = {'\u0652','\u06E1'}
SMALL_MEEM = {'\u06E2','\u06ED'}
NORM = str.maketrans({'أ':'ء','إ':'ء','آ':'ء','ؤ':'ء','ئ':'ء','ٱ':'ا','ى':'ي','ة':'ت'})
CATS = {'Izhar': set('ءهعحغخ'), 'Idgham': set('يرملون'), 'Iqlab': set('ب'),
        'Ikhfa': set('تثجدذزسشصضطظفقك')}
LABELS = ['Izhar', 'Idgham', 'Iqlab', 'Ikhfa']

def cat_of(ch, word_start):
    if ch == 'ا' and word_start: ch = 'ء'
    for k, s in CATS.items():
        if ch in s: return k
    return None

def tokens(text):
    t = unicodedata.normalize('NFC', str(text)).translate(NORM)
    toks, ws = [], True
    for ch in t:
        if ch in DIAC:
            if toks: toks[-1][1].add(ch)
        elif '\u0620' <= ch <= '\u064A' or '\u0671' <= ch <= '\u06D3':
            toks.append([ch, set(), ws]); ws = False
        else:
            ws = True
    return toks

def rule_feats(text):
    f = {}
    tk = tokens(text)
    for i in range(len(tk) - 1):
        c, d, _ = tk[i]
        j = i + 1
        if d & TANWEEN and tk[j][0] in 'اي' and not tk[j][2] and j + 1 < len(tk):
            j += 1                                   # skip silent alif/ya after tanween
        n = tk[j]
        cat = cat_of(n[0], n[2])
        if cat is None: continue
        if d & TANWEEN: f['tan_'+cat] = f.get('tan_'+cat, 0) + 1
        if c == 'م' and n[0] == 'ب' and n[2] and not (d & VOWELS):
            f['mb_Iqlab'] = f.get('mb_Iqlab', 0) + 1
        if c == 'ن':
            if d & SMALL_MEEM: f['mark_Iqlab'] = f.get('mark_Iqlab', 0) + 1
            if d & SUKUN: k = 'nsuk_'+cat
            elif d & (VOWELS | TANWEEN): continue
            elif n[2]: k = 'nend_'+cat
            else: k = 'nmid_'+cat
            f[k] = f.get(k, 0) + 1
    return f

W = {'tan': 3, 'nsuk': 3, 'mark': 3, 'nend': 2, 'nmid': 1, 'mb': 2}
def rule_predict(text):
    sc = dict.fromkeys(LABELS, 0.0)
    for k, v in rule_feats(text).items():
        kind, cat = k.split('_'); sc[cat] += W[kind] * v
    return max(sc, key=sc.get) if max(sc.values()) > 0 else None

def plain(text):   # text without diacritics
    return ''.join(ch for ch in unicodedata.normalize('NFC', str(text)).translate(NORM) if ch not in DIAC)

from sklearn.metrics import f1_score
rp = train.text.map(rule_predict)
print('rule fires on', rp.notna().mean().round(3), 'of train clips')
print('rule-only macro F1:', round(f1_score(train.label, rp.fillna('Ikhfa'), average='macro'), 4))

## 5. Text model: rule features + character n-grams → Logistic Regression
5-fold stratified cross-validation on the transcripts.

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline, make_union, FunctionTransformer

def F_rules(texts): return [rule_feats(t) for t in texts]
def F_plain(texts): return [plain(t) for t in texts]

def build():
    return make_pipeline(
        make_union(
            make_pipeline(FunctionTransformer(F_rules), DictVectorizer()),
            make_pipeline(FunctionTransformer(F_plain),
                          TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True))),
        LogisticRegression(C=3, max_iter=3000, class_weight='balanced'))

skf = StratifiedKFold(5, shuffle=True, random_state=42)
X, y = train.text.values, train.label.values
p = cross_val_predict(build(), X, y, cv=skf)
print('rules + text LR, 5-fold CV macro F1:', round(f1_score(y, p, average='macro'), 4))

model = build().fit(X, y)
pred = pd.Series(model.predict(test.text.values), index=test.index)

## 6. Rule-first prediction
Where the rule engine finds evidence, trust the rule; otherwise use the model.

In [ ]:
rt = test.text.map(rule_predict)
print('rule fires on', rt.notna().mean().round(3), 'of test clips')
final = pred.copy()
final[rt.notna()] = rt[rt.notna()]

## 7. Hard clips: re-transcribe with a larger ASR model
For test clips where the rule found nothing, transcribe again with `openai/whisper-large-v3` (first 30 s, one clip at a time to fit GPU memory) and apply the rule to the new text.

In [ ]:
del asr; gc.collect(); torch.cuda.empty_cache()
asr2 = pipeline('automatic-speech-recognition', model='openai/whisper-large-v3', device=device,
                torch_dtype=torch.float16 if device == 0 else torch.float32)
gen2 = {'language': 'ar', 'task': 'transcribe', 'num_beams': 1, 'max_new_tokens': 80}

hard = test.index[rt.isna()]
texts2 = []
for i in test.loc[hard, 'id']:
    try:
        y_ = librosa.load(path('test', i), sr=SR)[0][:SR * 30]
        texts2.append(asr2({'raw': y_, 'sampling_rate': SR}, generate_kwargs=gen2)['text'])
    except Exception:
        texts2.append(''); torch.cuda.empty_cache()

r2 = pd.Series([rule_predict(t) for t in texts2], index=hard)
print('large model resolved', r2.notna().mean().round(3), 'of hard clips')
final[r2.dropna().index] = r2.dropna()

## 8. Submission

In [ ]:
submission = pd.DataFrame({'id': test.id, 'label': final.values})
assert set(submission.id) == set(sample.id) and submission.label.isin(LABELS).all()
submission.to_csv('submission.csv', index=False)
print(submission.label.value_counts())